In [1]:
import sys
from pathlib import Path
print(sys.executable)
assert Path(sys.prefix).resolve() == Path('/home/miakhlov/course-notebooks/.venv').resolve()

/home/miakhlov/course-notebooks/.venv/bin/python


# ЛР 10. Реальный redocking 1IEP

**Сущность:** Выполнить официальный пример Vina и сравнить геометрию с подготовленным лигандом.

**Данные:** Открытые 1IEP PDBQT/SDF, box и официальный выход; upstream commit и hash — в docking/provenance.json.

**Workflow:** Проверка данных → box → Vina seed42/cpu1 → mapped RMSD → интерпретация.

Перед запуском прочитайте [guideline](../guides/lab10.md) и [теорию](../../Module_theory/Module%2010_docking.md).

## 1. Прогноз

Запишите ожидаемое поведение и отрицательный контроль. Какое изменение входа должно изменить результат, а какое — сохранить?

In [2]:
from pathlib import Path
import sys, json, math, csv, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# WHY: relative discovery makes the notebook portable across machines.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "Labs" / "chemistry.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Запустите notebook внутри репозитория курса")
sys.path.insert(0, str(ROOT / "Labs"))
DATA = ROOT / "Labs" / "datasets"
OUT = ROOT / "Labs" / "results" / "lab10"
OUT.mkdir(parents=True, exist_ok=True)
started = time.perf_counter()


## 2. Разбор и исполнение

1. Проверьте источник подготовленных файлов и систему координат.
2. Разберите официальный выход; при установленном Vina выполните собственный запуск.
3. Сравните score и RMSD по serial тяжёлых атомов, без независимого совмещения лиганда.

Ожидаемое: Полный запуск отмечен executed-vina. Score сравнивается с официальным примером описательно; RMSD конечен. Если бинарника нет, явно reference-analysis, а не фиктивный выполненный докинг.

In [3]:
from docking import run_docking
result=run_docking(ROOT,OUT)
print(json.dumps(result,indent=2))
assert math.isfinite(result["official_reference_rmsd_angstrom"])
assert result["reference_score"]<0
assert "rmsd_definition" in result
if result["mode"]=="executed-vina":
    assert math.isfinite(result["rmsd_angstrom"]) and result["score"]<0
else:
    print("Vina не установлен: выполнен только разбор официального выхода. Это явно ограниченный режим.")
artifact=result


{
  "official_reference_rmsd_angstrom": 0.906728765427707,
  "rmsd_definition": "mapped heavy atom serials, receptor frame, no symmetry optimization",
  "reference_score": -13.234,
  "mode": "executed-vina",
  "wall_seconds": 55.97170888400001,
  "score": -13.286,
  "rmsd_angstrom": 0.8968616151652141,
  "command": [
    "/home/miakhlov/.local/bin/vina",
    "--receptor",
    "/home/miakhlov/course-notebooks/Labs/datasets/docking/1iep_receptor.pdbqt",
    "--ligand",
    "/home/miakhlov/course-notebooks/Labs/datasets/docking/1iep_ligand.pdbqt",
    "--config",
    "/home/miakhlov/course-notebooks/Labs/datasets/docking/1iep_receptor.box.txt",
    "--seed",
    "42",
    "--cpu",
    "1",
    "--exhaustiveness",
    "8",
    "--num_modes",
    "9",
    "--out",
    "/home/miakhlov/course-notebooks/Labs/results/lab10/redocked.pdbqt"
  ],
  "binary_sha256": "f31f774f723bba7bbe6e9d1c47577020eea9a8da16424284c043d22593570644"
}


## 3. Проверяемая запись запуска

Benchmark фиксирует этот запуск. Время включает расчёт и создание артефактов, но не установку среды; это не сравнительный рейтинг оборудования.

In [4]:
elapsed = time.perf_counter()-started
run_record = {"lab": 10, "calculation_wall_seconds": elapsed, "result": artifact}
(OUT/"benchmark.json").write_text(json.dumps(run_record,ensure_ascii=False,indent=2),encoding="utf-8")
print("Запуск сохранён:",OUT/"benchmark.json")


Запуск сохранён: /home/miakhlov/course-notebooks/Labs/results/lab10/benchmark.json


## 4. Самостоятельная тренировка

1. Измените seed и сравните поиск при той же подготовке.
2. Почему serial-mapped RMSD не равен symmetry-corrected минимуму?
3. Какие шаги нужно добавить при подготовке нового сырого PDB?

[Ответы после попытки](../solutions/lab10.md). Вернитесь к первому заданию через неделю.

## 5. Ваш научный вывод

1. Какой результат получен?
2. Какой вывод из него пока не следует?
3. Какой следующий эксперимент или контроль нужен?

Сохраните свой вариант notebook отдельно; исходный пример должен остаться воспроизводимым.